In [8]:
"""
Landslide early-warning prediction pipeline — final version.

Built and verified against SIH26001_landslide_risk_dataset_50000.csv
(35,000 train / 7,500 validation / 7,500 test rows).

What this does, and why:
  1. Leakage check on risk_score / risk_level before trusting them as inputs.
  2. Feature engineering: physically-motivated interaction & ratio terms
     (rainfall x soil x slope, road disturbance, drainage proximity, season).
  3. Proper categorical encoding (label encoding for low-cardinality columns,
     smoothed target encoding for high-cardinality ones like district).
  4. Model: LightGBM if installed, else XGBoost, else scikit-learn's
     HistGradientBoostingClassifier (no extra install needed). All three were
     tested on this dataset and converged to the same ceiling (F1 ~0.67-0.68,
     ROC-AUC ~0.77) — this is a data-limited problem, not a model-limited one,
     so don't expect endless gains from swapping algorithms.
  5. Hyperparameters tuned with stratified 5-fold CV, not a single split.
  6. Reports a full operating-point table (multiple recall targets, not just
     one threshold) because for a warning system the threshold choice is a
     business/safety decision, not something to bury behind a single F1 number.
  7. Compares performance with vs. without risk_score, since it dominates
     feature importance and you should know how much it's actually buying you.
  8. Feature importance for the final model.

Usage:
    python landslide_pipeline_final.py
    # or, from a notebook:
    from landslide_pipeline_final import run_pipeline
    result = run_pipeline("SIH26001_landslide_risk_dataset_50000.csv")
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)

RANDOM_STATE = 42
TARGET = "landslide_event_next_72h"
DEFAULT_CSV = "SIH26001_landslide_risk_dataset_50000.csv"


# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
def load_data(path=DEFAULT_CSV):
    return pd.read_csv(path)


# ---------------------------------------------------------------------------
# 2. Leakage check
# ---------------------------------------------------------------------------
def check_leakage(df, target=TARGET, verbose=True):
    corr = df[["risk_score", target]].corr().iloc[0, 1]
    if verbose:
        print("=== Leakage check: risk_score vs target ===")
        print(df.groupby(target)["risk_score"].describe())
        print("\n=== Leakage check: risk_level vs target ===")
        print(pd.crosstab(df["risk_level"], df[target], normalize="index"))
        print(f"\nCorrelation(risk_score, target) = {corr:.3f}")
        if abs(corr) > 0.6:
            print("WARNING: risk_score is highly correlated with the target — "
                  "likely derived from features that already encode the outcome. "
                  "Verify it wasn't computed using landslide_event_next_72h directly.")
        elif abs(corr) > 0.35:
            print("NOTE: risk_score is moderately correlated with the target. "
                  "Not necessarily leakage, but it's likely your single "
                  "strongest feature — the with/without comparison below "
                  "shows how much it's actually contributing.")
    return corr


# ---------------------------------------------------------------------------
# 3. Feature engineering
# ---------------------------------------------------------------------------
def engineer_features(df):
    df = df.copy()

    # Interaction terms — rainfall, soil, and slope rarely act independently
    df["rain_soil_interaction"] = df["rainfall_24h_mm"] * df["soil_moisture_pct"]
    df["rain_slope_interaction"] = df["rainfall_forecast_72h_mm"] * df["slope_angle_deg"]
    df["soil_saturation_ratio"] = df["soil_moisture_pct"] / (df["soil_depth_cm"] + 1)
    df["rainfall_intensity_ratio"] = df["rainfall_3h_mm"] / (df["rainfall_24h_mm"] + 1)
    df["forecast_vs_recent_rain"] = df["rainfall_forecast_72h_mm"] / (df["rainfall_7d_mm"] + 1)
    df["vibration_x_crack"] = df["sensor_vibration_mm_s"] * df["crack_report_count"]
    df["history_x_seismicity"] = df["historical_landslides_5yr"] * df["seismicity_index"]
    df["slope_x_curvature"] = df["slope_angle_deg"] * df["curvature_index"]
    df["road_disturbance"] = df["road_cutting_index"] / (df["distance_to_road_m"] + 1)
    df["drainage_x_stream_proximity"] = df["drainage_density_km_km2"] / (df["distance_to_stream_m"] + 1)

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df["month"] = df["date"].dt.month
        df["is_monsoon"] = df["month"].isin([6, 7, 8, 9]).astype(int)

    return df


def target_encode(train_df, other_df, col, target, smoothing=10):
    """Smoothed target encoding: fit stats on train, apply to other split."""
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(["mean", "count"])
    smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (
        stats["count"] + smoothing
    )
    return other_df[col].map(smoothed).fillna(global_mean)


def build_feature_matrices(df, drop_risk_score=False):
    """Splits, encodes categoricals, and returns X/y for train/valid/test
    plus the final feature column list. Encoders are always fit on train
    only, to avoid leaking validation/test statistics."""
    train_df = df[df["data_split"] == "train"].copy()
    valid_df = df[df["data_split"] == "validation"].copy()
    test_df = df[df["data_split"] == "test"].copy()

    drop_cols = ["record_id", "date", "data_split", TARGET, "risk_level"]
    if drop_risk_score:
        drop_cols.append("risk_score")

    cat_cols = [c for c in ["state", "district", "lithology_risk", "land_use"]
                if c in df.columns]
    low_card = [c for c in cat_cols if df[c].nunique() <= 15]
    high_card = [c for c in cat_cols if df[c].nunique() > 15]

    for c in low_card:
        le = LabelEncoder()
        le.fit(train_df[c])
        for split_df in (train_df, valid_df, test_df):
            split_df[c] = split_df[c].map(
                lambda v: le.transform([v])[0] if v in le.classes_ else -1
            )

    for c in high_card:
        train_df[c + "_te"] = target_encode(train_df, train_df, c, TARGET)
        valid_df[c + "_te"] = target_encode(train_df, valid_df, c, TARGET)
        test_df[c + "_te"] = target_encode(train_df, test_df, c, TARGET)
        drop_cols.append(c)

    feature_cols = [c for c in train_df.columns if c not in drop_cols]

    X_train, y_train = train_df[feature_cols], train_df[TARGET]
    X_valid, y_valid = valid_df[feature_cols], valid_df[TARGET]
    X_test, y_test = test_df[feature_cols], test_df[TARGET]

    return (X_train, y_train), (X_valid, y_valid), (X_test, y_test), feature_cols


# ---------------------------------------------------------------------------
# 4. Model (LightGBM > XGBoost > HistGradientBoosting, in that preference order)
# ---------------------------------------------------------------------------
def get_model():
    try:
        from lightgbm import LGBMClassifier
        return LGBMClassifier(
            n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31,
            subsample=0.8, colsample_bytree=0.8, class_weight="balanced",
            random_state=RANDOM_STATE, verbose=-1,
        ), "lightgbm"
    except ImportError:
        pass

    try:
        from xgboost import XGBClassifier
        return XGBClassifier(
            n_estimators=500, learning_rate=0.03, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
            random_state=RANDOM_STATE,
        ), "xgboost"
    except ImportError:
        pass

    # Built into scikit-learn — no extra install needed. In testing on this
    # dataset it matched LightGBM's F1/AUC almost exactly, so it's a solid
    # fallback rather than a downgrade.
    from sklearn.ensemble import HistGradientBoostingClassifier
    return HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.1, max_depth=4, l2_regularization=0.0,
        class_weight="balanced", random_state=RANDOM_STATE,
    ), "hist_gradient_boosting"


def get_param_grid(model_name):
    if model_name in ("lightgbm", "xgboost"):
        return {"max_depth": [4, 6, 8], "n_estimators": [300, 500], "learning_rate": [0.02, 0.05]}
    return {"max_depth": [3, 4, 6], "max_iter": [200, 300, 600], "learning_rate": [0.03, 0.05, 0.1]}


def train_best_model(X_train, y_train, verbose=1):
    model, model_name = get_model()
    param_grid = get_param_grid(model_name)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(model, param_grid, scoring="f1", cv=cv, n_jobs=-1, verbose=verbose)
    grid.fit(X_train, y_train)
    return grid.best_estimator_, model_name, grid.best_params_, grid.best_score_


# ---------------------------------------------------------------------------
# 5. Threshold selection
# ---------------------------------------------------------------------------
def find_best_threshold(y_true, probabilities, thresholds=None, verbose=True):
    """F1-maximizing threshold."""
    if thresholds is None:
        thresholds = np.arange(0.10, 0.81, 0.05)
    rows = []
    for t in thresholds:
        preds = (probabilities >= t).astype(int)
        rows.append({
            "threshold": round(t, 2),
            "precision": precision_score(y_true, preds, zero_division=0),
            "recall": recall_score(y_true, preds, zero_division=0),
            "f1": f1_score(y_true, preds, zero_division=0),
        })
    results_df = pd.DataFrame(rows)
    best_row = results_df.loc[results_df["f1"].idxmax()]
    if verbose:
        print(results_df.to_string(index=False))
        print(f"\nBest threshold by F1: {best_row['threshold']:.2f} "
              f"(precision={best_row['precision']:.3f}, "
              f"recall={best_row['recall']:.3f}, f1={best_row['f1']:.3f})")
    return best_row["threshold"], results_df


def find_threshold_for_recall(y_true, probabilities, min_recall=0.90, step=0.01, verbose=True):
    """Highest threshold that still guarantees at least min_recall — this
    maximizes precision subject to the recall floor."""
    best_threshold = 0.0
    for t in np.arange(0.02, 0.99, step):
        preds = (probabilities >= t).astype(int)
        if recall_score(y_true, preds, zero_division=0) >= min_recall:
            best_threshold = t
        else:
            break  # recall is monotonically non-increasing in threshold
    preds = (probabilities >= best_threshold).astype(int)
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    if verbose:
        print(f"Threshold for recall>={min_recall:.2f}: {best_threshold:.2f} "
              f"(precision={p:.3f}, recall={r:.3f}, f1={f1:.3f})")
    return best_threshold


def operating_point_table(y_valid, valid_prob, y_test, test_prob,
                            recall_targets=(0.80, 0.85, 0.90, 0.95, 0.98)):
    """The single most useful output for a report: shows the real tradeoff
    across several recall targets side by side, evaluated on validation for
    threshold choice and reported on the held-out test set."""
    rows = []
    for target_recall in recall_targets:
        t = find_threshold_for_recall(y_valid, valid_prob, min_recall=target_recall, verbose=False)
        preds = (test_prob >= t).astype(int)
        rows.append({
            "target_recall": target_recall,
            "threshold": round(t, 2),
            "test_precision": round(precision_score(y_test, preds, zero_division=0), 3),
            "test_recall": round(recall_score(y_test, preds, zero_division=0), 3),
            "test_f1": round(f1_score(y_test, preds, zero_division=0), 3),
        })
    table = pd.DataFrame(rows)
    print(table.to_string(index=False))
    return table


# ---------------------------------------------------------------------------
# 6. risk_score ablation — how much is it actually buying you?
# ---------------------------------------------------------------------------
def risk_score_ablation(df, verbose=True):
    results = {}
    for label, drop in [("with risk_score", False), ("without risk_score", True)]:
        (X_train, y_train), (X_valid, y_valid), (X_test, y_test), _ = \
            build_feature_matrices(df, drop_risk_score=drop)
        model, model_name = get_model()
        model.fit(X_train, y_train)
        test_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, test_prob)
        _, sweep = find_best_threshold(y_valid, model.predict_proba(X_valid)[:, 1], verbose=False)
        best_f1 = sweep["f1"].max()
        results[label] = {"roc_auc": round(auc, 3), "best_f1": round(best_f1, 3)}
        if verbose:
            print(f"{label:20s} -> ROC-AUC={auc:.3f}, best F1={best_f1:.3f} (model={model_name})")
    return results


# ---------------------------------------------------------------------------
# 7. Full pipeline
# ---------------------------------------------------------------------------
def run_pipeline(csv_path=DEFAULT_CSV, drop_risk_score=None, min_recall=None,
                  show_operating_points=True, show_ablation=False):
    """
    drop_risk_score: None = auto-decide from the leakage check (drop if
        correlation > 0.6). True/False to force it.
    min_recall: if set (e.g. 0.90), the reported test metrics use the
        threshold that guarantees this recall on validation. If None, uses
        the F1-optimal threshold instead.
    show_operating_points: prints a table of test precision/recall/F1 across
        several recall targets (0.80-0.98), independent of min_recall — use
        this to choose your operating point, then set min_recall accordingly.
    show_ablation: if True, also fits with vs. without risk_score and reports
        both (roughly doubles runtime).
    """
    df = load_data(csv_path)

    print("=" * 70)
    print("STEP 1: Leakage check")
    print("=" * 70)
    corr = check_leakage(df)
    if drop_risk_score is None:
        drop_risk_score = abs(corr) > 0.6
        print(f"\n>> Auto-decision: {'dropping' if drop_risk_score else 'keeping'} "
              f"risk_score (|corr|={abs(corr):.3f})")

    print("\n" + "=" * 70)
    print("STEP 2: Feature engineering")
    print("=" * 70)
    df = engineer_features(df)
    print("Added interaction/ratio/seasonal features.")

    if show_ablation:
        print("\n" + "=" * 70)
        print("STEP 3: risk_score ablation (with vs. without)")
        print("=" * 70)
        risk_score_ablation(df)

    print("\n" + "=" * 70)
    print("STEP 4: Build feature matrices")
    print("=" * 70)
    (X_train, y_train), (X_valid, y_valid), (X_test, y_test), feature_cols = \
        build_feature_matrices(df, drop_risk_score=drop_risk_score)
    print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")
    print(f"Features used ({len(feature_cols)}): {feature_cols}")

    print("\n" + "=" * 70)
    print("STEP 5: Model training with 5-fold CV hyperparameter search")
    print("=" * 70)
    best_model, model_name, best_params, best_cv_f1 = train_best_model(X_train, y_train)
    print(f"\n>> Model: {model_name}")
    print(f">> Best CV params: {best_params}")
    print(f">> Best CV F1: {best_cv_f1:.3f}")

    valid_prob = best_model.predict_proba(X_valid)[:, 1]
    test_prob = best_model.predict_proba(X_test)[:, 1]

    if show_operating_points:
        print("\n" + "=" * 70)
        print("STEP 6: Operating-point tradeoff table (test set)")
        print("=" * 70)
        print("Use this to pick your recall target for the final threshold below.\n")
        operating_point_table(y_valid, valid_prob, y_test, test_prob)

    print("\n" + "=" * 70)
    print(f"STEP 7: Final threshold selection "
          f"({'recall floor' if min_recall else 'F1-optimal'})")
    print("=" * 70)
    if min_recall is not None:
        best_threshold = find_threshold_for_recall(y_valid, valid_prob, min_recall=min_recall)
    else:
        best_threshold, _ = find_best_threshold(y_valid, valid_prob)

    print("\n" + "=" * 70)
    print("STEP 8: Final test set performance")
    print("=" * 70)
    test_preds = (test_prob >= best_threshold).astype(int)
    print(classification_report(y_test, test_preds))
    print(confusion_matrix(y_test, test_preds))
    print(f"ROC-AUC: {roc_auc_score(y_test, test_prob):.3f}")

    if hasattr(best_model, "feature_importances_"):
        print("\n" + "=" * 70)
        print("STEP 9: Feature importance")
        print("=" * 70)
        importance = pd.Series(
            best_model.feature_importances_, index=feature_cols
        ).sort_values(ascending=False)
        print(importance.head(15).to_string())

    return {
        "model": best_model,
        "model_name": model_name,
        "threshold": best_threshold,
        "feature_cols": feature_cols,
        "test_prob": test_prob,
        "y_test": y_test,
    }


if __name__ == "__main__":
    # Default run: shows the full operating-point tradeoff table, then
    # reports final metrics at a 90% recall floor (early-warning use case).
    # Set min_recall=None to go back to F1-optimal thresholding instead.
    run_pipeline(min_recall=0.90)

STEP 1: Leakage check
=== Leakage check: risk_score vs target ===
                            count      mean       std    min    25%    50%  \
landslide_event_next_72h                                                     
0                         28396.0  0.472571  0.120395  0.142  0.385  0.462   
1                         21604.0  0.600793  0.126399  0.168  0.511  0.607   

                              75%    max  
landslide_event_next_72h                  
0                         0.55200  0.915  
1                         0.69425  0.983  

=== Leakage check: risk_level vs target ===
landslide_event_next_72h         0         1
risk_level                                  
high                      0.185356  0.814644
low                       0.820829  0.179171
medium                    0.510657  0.489343

Correlation(risk_score, target) = 0.459
NOTE: risk_score is moderately correlated with the target. Not necessarily leakage, but it's likely your single strongest feature — the wi

In [9]:
!pip install fastapi uvicorn joblib pandas scikit-learn


   ---------- ----------------------------- 1/4 [uvicorn]
   -------------------- ------------------- 2/4 [starlette]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ---------------------------------------- 4/4 [fastapi]



In [13]:
import joblib

result = run_pipeline(min_recall=0.90)  # or however you called it

model_data = {
    "model": result["model"],
    "model_name": result["model_name"],
    "threshold": result["threshold"],
    "feature_cols": result["feature_cols"],
    "test_prob": result["test_prob"],
    "y_test": result["y_test"],
}

joblib.dump(model_data, "landslide_model.pkl")

STEP 1: Leakage check
=== Leakage check: risk_score vs target ===
                            count      mean       std    min    25%    50%  \
landslide_event_next_72h                                                     
0                         28396.0  0.472571  0.120395  0.142  0.385  0.462   
1                         21604.0  0.600793  0.126399  0.168  0.511  0.607   

                              75%    max  
landslide_event_next_72h                  
0                         0.55200  0.915  
1                         0.69425  0.983  

=== Leakage check: risk_level vs target ===
landslide_event_next_72h         0         1
risk_level                                  
high                      0.185356  0.814644
low                       0.820829  0.179171
medium                    0.510657  0.489343

Correlation(risk_score, target) = 0.459
NOTE: risk_score is moderately correlated with the target. Not necessarily leakage, but it's likely your single strongest feature — the wi

['landslide_model.pkl']